## 7. `@property` & Decorators

### `@property`
The `@property` decorator lets you access a **method as if it were an attribute** — no parentheses needed. It enables **computed attributes** and **controlled attribute access** (getters, setters, deleters) without changing the public API.

```python
class Circle:
    @property
    def diameter(self):
        return self.radius * 2    # accessed as  obj.diameter  not  obj.diameter()

    @diameter.setter
    def diameter(self, value):
        self.radius = value / 2

    @diameter.deleter
    def diameter(self):
        del self.radius
```

### Why use `@property` over direct attributes?
| Reason | Example |
|--------|---------|
| Add validation on set | Reject negative radius |
| Make attribute read-only | No setter defined → `AttributeError` on assignment |
| Compute on-the-fly | `age` calculated from `birth_year` |
| Change internal implementation without breaking API | Rename `_r` without users noticing |

### Name mangling & privacy conventions
| Prefix | Convention | Behaviour |
|--------|------------|----------|
| `name` | Public | Accessible anywhere |
| `_name` | Protected (convention) | "Don't touch this" — still accessible |
| `__name` | Private (name-mangled) | Renamed to `_ClassName__name` internally |

In [1]:
# ---- @property — validated temperature class ----
class Temperature:
    """Stores temperature in Celsius with validation and computed Fahrenheit."""

    def __init__(self, celsius=0):
        self.celsius = celsius     # uses the setter immediately

    @property
    def celsius(self):
        """Get temperature in Celsius."""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        """Set temperature — rejects values below absolute zero."""
        if value < -273.15:
            raise ValueError(f"Temperature {value}°C is below absolute zero!")
        self._celsius = value

    @property
    def fahrenheit(self):
        """Computed read-only property — no setter defined."""
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        """Allow setting via Fahrenheit — converts to Celsius."""
        self.celsius = (value - 32) * 5/9

    @property
    def kelvin(self):
        return self._celsius + 273.15

    def __repr__(self):
        return f"Temperature({self._celsius}°C / {self.fahrenheit:.1f}°F / {self.kelvin:.2f}K)"


t = Temperature(25)
print(t)

t.celsius = 100
print(t)

t.fahrenheit = 32      # set via Fahrenheit — converts internally
print(t)

try:
    t.celsius = -300   # below absolute zero
except ValueError as e:
    print(f"ValueError: {e}")

Temperature(25°C / 77.0°F / 298.15K)
Temperature(100°C / 212.0°F / 373.15K)
Temperature(0.0°C / 32.0°F / 273.15K)
ValueError: Temperature -300°C is below absolute zero!


In [2]:
# ---- Privacy conventions ----
class SecureAccount:
    def __init__(self, owner, pin):
        self.owner  = owner          # public
        self._log   = []             # protected — internal use
        self.__pin  = pin            # private — name mangled to _SecureAccount__pin

    def verify_pin(self, entered):
        success = (entered == self.__pin)
        self._log.append(f"PIN attempt: {'OK' if success else 'FAIL'}")
        return success

    @property
    def log(self):
        """Read-only view of the log."""
        return list(self._log)       # return a copy — caller can't mutate internal list


acc = SecureAccount("Purvi", "1234")
print(acc.verify_pin("0000"))   # False
print(acc.verify_pin("1234"))   # True
print(acc.log)

# Name mangling
try:
    print(acc.__pin)             # AttributeError!
except AttributeError as e:
    print(f"AttributeError: {e}")

print(acc._SecureAccount__pin)  # mangled name — accessible but signals "don't do this"

False
True
['PIN attempt: FAIL', 'PIN attempt: OK']
AttributeError: 'SecureAccount' object has no attribute '__pin'
1234


In [3]:
# ---- Caching with @property (compute once) ----
class DataProcessor:
    """Demonstrates caching an expensive computation."""

    def __init__(self, data):
        self.data = data
        self._stats = None          # cache sentinel

    @property
    def stats(self):
        if self._stats is None:     # compute only once
            print("  [computing stats...]")
            n = len(self.data)
            self._stats = {
                "count": n,
                "mean":  sum(self.data) / n,
                "min":   min(self.data),
                "max":   max(self.data),
            }
        return self._stats

dp = DataProcessor([4, 7, 2, 9, 1, 5, 8, 3, 6])
print(dp.stats)   # computed
print(dp.stats)   # from cache — no recompute message

  [computing stats...]
{'count': 9, 'mean': 5.0, 'min': 1, 'max': 9}
{'count': 9, 'mean': 5.0, 'min': 1, 'max': 9}
